In [15]:
import pandas as pd
import numpy as np
import os

from score import compute_pbf_score

labels = {
    'left': -1,
    'center': 0,
    'right': 1
}
    
input_dir = '../summaries/'

def split_pred_by_stance(df_steered, k=3): 
    '''
    Splits predictions in given dataframe by steering stance, 
    which repeats in a sequence of 'left', 'right', and 'center's. 
    '''
    stances = {'left': [], 'right': [], 'center': []}
    
    num_summaries_per_stance = df_steered.shape[0] // k
    assert num_summaries_per_stance == 900, "expecting 900 summaries per stance"
    
    for i in range(num_summaries_per_stance): 
        stance = ['left', 'right', 'center'][i % 3]
        pred = [labels[df_steered.iloc[i*k + j]['Predicted Bias']] for j in range(k)]
        stances[stance].append(pred)
    
    left, right, center = stances['left'], stances['right'], stances['center']
    len(left) == len(right) == len(center) == 300, "expecting 300 (groups of) predictions per stance"

    return left, right, center

def split_pred_by_matching(df_steered, source_bias, k=3):
    """
    Splits predictions into two categories based on whether their 
    steer stance match the source bias.
    """
    match_source, match_pred = [], []
    mismatch_source, mismatch_pred = [], []

    num_summaries_per_stance = df_steered.shape[0] // k

    for i in range(num_summaries_per_stance):
        pred = [labels[df_steered.iloc[i * k + j]['Predicted Bias']] for j in range(k)]
        steer_stance = labels[df_steered.iloc[i * k]['Stance']]
        source_stance = source_bias[i//k]

        if steer_stance == source_stance:
            match_source.append(source_stance)
            match_pred.append(pred)
        else:
            mismatch_source.append(source_stance)
            mismatch_pred.append(pred)

    return match_source, match_pred, mismatch_source, mismatch_pred
    
    
def get_stances(df, k=3): 
    true, pred = [], []
    
    for i in range(0, df.shape[0], k):
        true.append(labels[df.iloc[i]['Stance']])
        group = [labels[df.iloc[i+j]['Predicted Bias']] for j in range(k)]
        pred.append(group)
        
    return true, pred


def matches(lst1, lst2):
    return sum(y.count(x) for x, y in zip(lst1, lst2)) / (len(lst1) * 3)


def process_summaries(model, k=3):
    if model not in ['bart', 't5', 'gpt2', 'neo', 'llama']: 
        return 
    
    print(f'Processing summaries for {model}...')
    
    path = os.path.join(input_dir, f'{model}.csv')    
    df = pd.read_csv(path)
    print(f'\tLoaded {path} with {df.shape[0]} summaries.')
    
    path_steered  = os.path.join(input_dir, f'{model}-prompt.csv')
    df_steered = pd.read_csv(path_steered)
    print(f'\tLoaded {path_steered} with {df_steered.shape[0]} summaries.')
    
    print('\n')
    
    # without steering intervention
    true, pred = get_stances(df)
    pbf = compute_pbf_score(np.array(true), np.array(pred))
    print(f' - percentage matching with steering: {matches(true,pred)}')
    print(f' - PBF score without steering: {pbf}')
    
    print('\n')
    
    # with steering intervention
    target, pred_steered = get_stances(df_steered)
    pbf_steered = compute_pbf_score(np.array(target), np.array(pred_steered))
    print(f' - percentage matching with steering (ALL): {matches(target,pred_steered)}')
    print(f' - PBF score with steering (ALL): {pbf_steered}')
    
    pred_left, pred_right, pred_center = split_pred_by_stance(df_steered)

    left = [labels['left']]*len(pred_left)
    pbf_left = compute_pbf_score(np.array(left), np.array(pred_left))
    print(f'\t - percentage matching with steering (LEFT): {matches(left,pred_left)}')
    print(f'\t - PBF score with steering (LEFT): {pbf_left}')
    
    center = [labels['center']]*len(pred_center)
    pbf_center = compute_pbf_score(np.array(center), np.array(pred_center))
    print(f'\t - percentage matching with steering (CENTER): {matches(center,pred_center)}')
    print(f'\t - PBF score with steering (CENTER): {pbf_center}')
    
    right = [labels['right']]*len(pred_right)
    pbf_right = compute_pbf_score(np.array(right), np.array(pred_right))
    print(f'\t - percentage matching with steering (RIGHT): {matches(right,pred_right)}')
    print(f'\t - PBF score with steering (RIGHT): {pbf_right}')
    
    print('\n')
    
    source = [labels[df.iloc[i]['Stance']] for i in range(0, df.shape[0], k)]
    match_source, match_pred, mismatch_source, mismatch_pred = split_pred_by_matching(df_steered, source)
    
    pbf_match = compute_pbf_score(np.array(match_source), np.array(match_pred))
    print(f' - PBF score with steering, where target bias matches source bias: {pbf_match}')
    
    pbf_mismatch = compute_pbf_score(np.array(mismatch_source), np.array(mismatch_pred))
    print(f' - PBF score with steering, where target bias does *not* match source bias: {pbf_mismatch}')
    
    print('\n--------------------\n')

In [17]:
for model in ['bart', 't5', 'gpt2', 'neo', 'llama']: 
    process_summaries(model)

Processing summaries for bart...
	Loaded ../summaries/bart.csv with 900 summaries.
	Loaded ../summaries/bart-prompt.csv with 2700 summaries.


 - percentage matching with steering: 0.4022222222222222
 - PBF score without steering: 0.5253656378033953


 - percentage matching with steering (ALL): 0.3388888888888889
 - PBF score with steering (ALL): 0.45993827513267826
	 - percentage matching with steering (LEFT): 0.17222222222222222
	 - PBF score with steering (LEFT): 0.3295938212960412
	 - percentage matching with steering (CENTER): 0.5166666666666667
	 - PBF score with steering (CENTER): 0.6523891064230964
	 - percentage matching with steering (RIGHT): 0.3277777777777778
	 - PBF score with steering (RIGHT): 0.44798349461069353


 - PBF score with steering, where target bias matches source bias: 0.5283009433971698
 - PBF score with steering, where target bias does *not* match source bias: 0.5272715132105723

--------------------

Processing summaries for t5...
	Loaded ../summaries/t5.cs